# Modelo B — notebook maestro productivo A9

Orquestación Colab para el entrenamiento productivo congelado de Modelo B. Preflight y diagnóstico no entrenan; la cola requiere una acción explícita.

In [ ]:
# PARÁMETROS OPERATIVOS — única celda editable
from pathlib import Path
import re

REPO_URL = 'https://github.com/sap15/ViVU_lab.git'
GIT_COMMIT = '3757bfdde52a9920611f3e7b8027434ce80d54bc'
ARCHITECTURE = 'model_b_graph_level_relational'
RUN_SEEDS = [11, 23, 37, 41, 53]
SPLIT_SEED = 42
REQUESTED_DEVICE = 'cuda'

DRIVE_PROJECT_BASE = '/content/drive/MyDrive/modelos_proyecto_PKP2'
DRIVE_MUTANTS_HDF5 = (
    DRIVE_PROJECT_BASE
    + '/model_a/data/proc_483p.hdf5'
)
DRIVE_WT_HDF5 = (
    DRIVE_PROJECT_BASE
    + '/model_a/data/wt_companion.hdf5'
)
DRIVE_MODEL_A_A9_ROOT = DRIVE_PROJECT_BASE + '/model_a/runs/model_a_a9'
DRIVE_MODEL_B_A9_ROOT = DRIVE_PROJECT_BASE + '/model_b/runs/model_b_a9'

LOCAL_ROOT = '/content/model_b_workspace'
REPO_DIR = LOCAL_ROOT + '/repo'
STAGING_ROOT = LOCAL_ROOT + '/staging'
RUN_PRODUCTIVE_QUEUE = False

EXPECTED_MUTANTS_SHA256 = '92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631'
EXPECTED_WT_SHA256 = '29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a'

if RUN_SEEDS != [11, 23, 37, 41, 53] or SPLIT_SEED != 42:
    raise ValueError('Contrato A9: RUN_SEEDS y SPLIT_SEED están congelados')
if REQUESTED_DEVICE != 'cuda' or ARCHITECTURE != 'model_b_graph_level_relational':
    raise ValueError('Contrato A9: CUDA y arquitectura B canónica obligatorias')
if not re.fullmatch(r'[0-9a-f]{40}', GIT_COMMIT):
    raise ValueError('GIT_COMMIT debe ser SHA completo')


## 1. Montaje de Drive y validación de locators inmutables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_project = Path(DRIVE_PROJECT_BASE).resolve()
if not drive_project.is_dir():
    raise FileNotFoundError(f'DRIVE_PROJECT_BASE inexistente: {drive_project}')
if not DRIVE_MUTANTS_HDF5 or not DRIVE_WT_HDF5:
    raise ValueError('Defina los locators Drive reales de ambos HDF5 en PARÁMETROS OPERATIVOS')
for label, raw in (('mutants', DRIVE_MUTANTS_HDF5), ('WT companion', DRIVE_WT_HDF5)):
    path = Path(raw).resolve()
    if not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f'HDF5 {label} ausente o vacío: {path}')


## 2. Clon controlado y checkout detached del commit congelado

In [ ]:
import subprocess, sys
local_root = Path(LOCAL_ROOT).resolve(); repo = Path(REPO_DIR).resolve()
if repo == local_root or not repo.is_relative_to(local_root):
    raise ValueError('REPO_DIR debe ser hijo de LOCAL_ROOT')
local_root.mkdir(parents=True, exist_ok=True)
fresh_clone = not (repo / '.git').is_dir()
if fresh_clone:
    if repo.exists():
        raise RuntimeError('REPO_DIR existe pero no es el clon controlado; no se borrará')
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo)], check=True)
sys.path[:0] = [str(repo), str(repo / 'src')]
from scripts.colab_preflight import checkout_git_revision, git_revision
GIT_INFO = checkout_git_revision(repo, GIT_COMMIT, expected_remote_url=REPO_URL, fresh_clone=fresh_clone)
git_revision(repo, GIT_COMMIT)
if GIT_INFO['commit'] != GIT_COMMIT:
    raise RuntimeError('El checkout no coincide con el commit A9 congelado')
print(GIT_INFO)


## 3. Entorno Colab y CUDA productiva obligatoria

In [ ]:
import platform, torch
from scripts.colab_preflight import prepare_colab_environment, runtime_summary
ENVIRONMENT = prepare_colab_environment(repo_root=repo, marker_root=local_root, commit=GIT_COMMIT, requirements_path=repo / 'requirements-colab.txt', device='cuda', python_version=platform.python_version(), torch_version=torch.__version__)
RUNTIME = runtime_summary('cuda')
if not RUNTIME['cuda_visible'] or RUNTIME['selected_device'] != 'cuda':
    raise RuntimeError('B_PRODUCTIVE_GPU=FAIL: CUDA no disponible; no se iniciará entrenamiento')
print({'B_PRODUCTIVE_GPU': 'PASS', **RUNTIME})


## 4. Staging local: hash, copia verificada y HDF5 solo lectura

In [ ]:
from scripts.colab_preflight import stage_file, require_free_space
staging = Path(STAGING_ROOT).resolve(); staging.mkdir(parents=True, exist_ok=True)
required_bytes = Path(DRIVE_MUTANTS_HDF5).stat().st_size + Path(DRIVE_WT_HDF5).stat().st_size
require_free_space(staging, required_bytes * 2)
MUTANTS_RECORD = stage_file(DRIVE_MUTANTS_HDF5, staging / 'proc_483p.hdf5', staging_root=staging, role='mutants')
WT_RECORD = stage_file(DRIVE_WT_HDF5, staging / 'wt_companion.hdf5', staging_root=staging, role='wt_companion')
MUTANTS_SHA256, WT_SHA256 = MUTANTS_RECORD['sha256'], WT_RECORD['sha256']
if MUTANTS_SHA256 != EXPECTED_MUTANTS_SHA256 or WT_SHA256 != EXPECTED_WT_SHA256:
    raise RuntimeError('HDF5 fingerprint mismatch: inputs científicos congelados rechazados')
for record in (MUTANTS_RECORD, WT_RECORD):
    Path(record['local_locator']).chmod(0o444)
if any(Path(record['local_locator']).stat().st_mode & 0o222 for record in (MUTANTS_RECORD, WT_RECORD)):
    raise RuntimeError('HDF5 staging no quedó en solo lectura')
STAGING_STATUS = 'PASS_READ_ONLY'
print({'MUTANTS_SHA256': MUTANTS_SHA256, 'WT_SHA256': WT_SHA256, 'STAGING_STATUS': STAGING_STATUS})


## 5. Configuración B A9 resuelta y preflight productivo único

Reutiliza el resolver oficial A9: no materializa ni crea un split nuevo.

In [ ]:
import json, os
os.chdir(repo)
from scripts.a9_preflight import resolve_a9_runtime_configs, validate_a9_colab_preflight
from gnn_siamese.config import save_config
from gnn_siamese.training.a9_contract import validate_separate_output_roots
A_OUTPUT_ROOT = Path(DRIVE_MODEL_A_A9_ROOT).resolve()
B_OUTPUT_ROOT = Path(DRIVE_MODEL_B_A9_ROOT).resolve()
ROOTS = validate_separate_output_roots(A_OUTPUT_ROOT, B_OUTPUT_ROOT)
FROZEN_SPLIT = (
    repo
    / 'splits'
    / 'leave_position_out_seed_42.json'
).resolve()
if not FROZEN_SPLIT.is_file():
    raise FileNotFoundError(f'FROZEN_SPLIT inexistente: {FROZEN_SPLIT}')
resolved_dir = local_root / 'resolved_configs'; resolved_dir.mkdir(parents=True, exist_ok=True)
config_a, config_b = resolve_a9_runtime_configs(repo / 'configs/model_b_a9.yaml', architecture=ARCHITECTURE, run_seed=RUN_SEEDS[0], mutants_hdf5=MUTANTS_RECORD['local_locator'], wt_hdf5=WT_RECORD['local_locator'], output_root=B_OUTPUT_ROOT, peer_output_root=A_OUTPUT_ROOT, repo_root=repo)
if (
    Path(config_b['split']['persist_path']).resolve() != FROZEN_SPLIT
    or config_b['split']['seed'] != 42
    or config_b['split']['allow_create'] is not False
):
    raise RuntimeError('B_PREFLIGHT_SPLIT=FAIL: split no está congelado')
RESOLVED_CONFIG = resolved_dir / 'model_b_a9_seed11.yaml'
save_config(config_b, RESOLVED_CONFIG)
PREFLIGHT = validate_a9_colab_preflight(repo / 'configs/model_b_a9.yaml', architecture=ARCHITECTURE, run_seed=RUN_SEEDS[0], mutants_hdf5=MUTANTS_RECORD['local_locator'], wt_hdf5=WT_RECORD['local_locator'], output_root=B_OUTPUT_ROOT, peer_output_root=A_OUTPUT_ROOT, repo_root=repo, expected_commit=GIT_COMMIT, allowed_output_root=drive_project)
if PREFLIGHT['status'] != 'PASS':
    raise RuntimeError('B productive preflight did not pass')
print(json.dumps({'B_PREFLIGHT': PREFLIGHT['status'], 'split_seed': SPLIT_SEED, 'partitions': PREFLIGHT['inventories']['model_b']['partitions'], 'A_OUTPUT_ROOT': ROOTS['model_a'], 'B_OUTPUT_ROOT': ROOTS['model_b']}, indent=2))


## 6. Supervisor persistente de cola B

El helper vive fuera del clon. Sólo se ejecuta desde la celda deliberada posterior.

In [ ]:
QUEUE_ROOT = B_OUTPUT_ROOT / '_queue'
LAUNCHER_LOG_ROOT = QUEUE_ROOT / '_launcher_logs'
QUEUE_ROOT.mkdir(parents=True, exist_ok=True); LAUNCHER_LOG_ROOT.mkdir(parents=True, exist_ok=True)
QUEUE_STATE = QUEUE_ROOT / 'model_b_a9_multiseed_queue_state.json'
QUEUE_LAUNCHER = local_root / 'model_b_a9_multiseed_queue_launcher.py'
helper = r'''import json, math, os, subprocess, sys, traceback
from datetime import datetime, timezone
from pathlib import Path
import yaml
def now(): return datetime.now(timezone.utc).isoformat()
def write(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True); temporary=path.with_suffix(path.suffix+'.tmp'); temporary.write_text(json.dumps(payload,indent=2,sort_keys=True,default=str),encoding='utf-8'); os.replace(temporary,path)
def load(path):
    try: return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc: raise RuntimeError(f'Unreadable JSON {path}: {exc}') from exc
def field(value,*parts):
    for part in parts:
        if not isinstance(value,dict) or part not in value: raise RuntimeError('missing field: '+'.'.join(parts))
        value=value[part]
    return value
def hits(root,seed,architecture):
    found=[]
    for path in root.rglob('run_manifest.json'):
        manifest=load(path)
        if int(field(manifest,'configuration','seed')) == seed:
            if manifest.get('architecture') != architecture: raise RuntimeError(f'incompatible architecture at {path}')
            found.append((path.parent,manifest))
    if len(found)>1: raise RuntimeError(f'AMBIGUOUS_B_RUNS: seed {seed} has {len(found)} manifests')
    return found
def gate(run_dir,seed,architecture,FROZEN_SPLIT):
    manifest=load(run_dir/'run_manifest.json')
    if manifest.get('status')!='completed' or manifest.get('architecture')!=architecture: raise RuntimeError('completed manifest mismatch')
    if int(field(manifest,'configuration','seed'))!=seed or int(field(manifest,'configuration','seed_bundle','split'))!=42: raise RuntimeError('seed contract mismatch')
    for path in (run_dir/'config_resolved.yaml',run_dir/'split.json',run_dir/'gradient_audit.json',run_dir/'metrics.jsonl',run_dir/'checkpoints'/'best.pt',run_dir/'checkpoints'/'last.pt'):
        if not path.is_file() or path.stat().st_size==0: raise RuntimeError(f'missing or empty {path}')
    config=yaml.safe_load((run_dir/'config_resolved.yaml').read_text(encoding='utf-8'))
    if config['model']['architecture']!=architecture or config['split']['allow_create'] is not False or int(config['split']['seed'])!=42 or Path(config['split']['persist_path']).resolve()!=FROZEN_SPLIT: raise RuntimeError('resolved config incompatible')
    audit=load(run_dir/'gradient_audit.json')
    from gnn_siamese.training.gradient_audit import expected_a9_active_module_names
    expected=expected_a9_active_module_names(config)
    if not audit or any(name not in audit for name in expected): raise RuntimeError('gradient audit incompatible with active B modules')
    metrics=[json.loads(line) for line in (run_dir/'metrics.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    if not metrics: raise RuntimeError('metrics.jsonl empty')
    epochs=[]
    for row in metrics:
        try:
            epoch=int(row['epoch']); train_loss=float(row['train']['mean_loss']); val_loss=float(row['validation']['mean_loss'])
        except (KeyError,TypeError,ValueError) as exc: raise RuntimeError('METRICS_SCHEMA_MISMATCH') from exc
        if not (math.isfinite(train_loss) and math.isfinite(val_loss)): raise RuntimeError(f'nonfinite loss in metrics for epoch {row.get("epoch")}')
        epochs.append(epoch)
    epochs_completed=int(manifest['training']['epochs_completed']); metric_records=len(metrics)
    if epochs_completed!=metric_records: raise RuntimeError('epochs_completed != metric_records: ' f'{epochs_completed} vs {metric_records}')
    if len(epochs)!=len(set(epochs)): raise RuntimeError('duplicate epochs in metrics.jsonl')
    if epochs_completed<1: raise RuntimeError('epochs_completed incoherent')
    global_step=manifest.get('training',{}).get('global_step')
    if global_step is not None and int(global_step)<=0: raise RuntimeError('global_step incoherent')
    return {'epochs_completed':epochs_completed,'metric_records':metric_records}
def active_competing_processes():
    result = subprocess.run(
        [
            'ps',
            '-eo',
            'pid=,args=',
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    current_pid = os.getpid()

    trainers = []
    other_launchers = []

    for line in (
        result.stdout.splitlines()
    ):

        stripped = line.strip()

        if not stripped:
            continue

        parts = stripped.split(
            maxsplit=1
        )

        if len(parts) != 2:
            continue

        try:
            pid = int(
                parts[0]
            )
        except ValueError:
            continue

        command = parts[1]

        if pid == current_pid:
            continue

        if (
            'scripts/train.py'
            in command
        ):

            trainers.append(
                stripped
            )

        if (
            'model_b_a9_multiseed_'
            'queue_launcher.py'
            in command
        ):

            other_launchers.append(
                stripped
            )

    return (
        trainers,
        other_launchers,
    )


def assert_no_competing_processes(
    stage,
):

    trainers, other_launchers = (
        active_competing_processes()
    )

    if trainers or other_launchers:

        raise RuntimeError(
            'B_QUEUE_PROCESS_GUARD_FAIL '
            f'at {stage}: '
            f'trainers={trainers}; '
            f'other_launchers='
            f'{other_launchers}'
        )


def main():
    payload=load(Path(sys.argv[1])); repo=Path(payload['repo']).resolve()
    os.chdir(repo)
    for candidate in (str(repo),str(repo/'src')):
        if candidate not in sys.path: sys.path.insert(0,candidate)
    from scripts.a9_preflight import resolve_a9_runtime_configs
    from gnn_siamese.config import save_config
    from gnn_siamese.training.gradient_audit import expected_a9_active_module_names
    del expected_a9_active_module_names
    assert_no_competing_processes(
        'launcher_startup'
    )
    root=Path(payload['output_root']).resolve(); state=Path(payload['state']).resolve(); architecture=payload['architecture']
    FROZEN_SPLIT=(repo/'splits'/'leave_position_out_seed_42.json').resolve()
    if not FROZEN_SPLIT.is_file(): raise FileNotFoundError(f'FROZEN_SPLIT inexistente: {FROZEN_SPLIT}')
    for seed in payload['seeds']:
        existing=hits(root,seed,architecture)
        if existing:
            run_dir,manifest=existing[0]; status=manifest.get('status')
            if status=='completed': result=gate(run_dir,seed,architecture,FROZEN_SPLIT)
            elif status=='running': raise RuntimeError(f'RECOVERY_REQUIRED: running seed {seed} at {run_dir}')
            else: raise RuntimeError(f'RECOVERY_REQUIRED: seed {seed} status {status!r}; no overwrite')
        else:
            _a,config=resolve_a9_runtime_configs(repo/'configs/model_b_a9.yaml',architecture=architecture,run_seed=seed,mutants_hdf5=payload['mutants'],wt_hdf5=payload['wt'],output_root=root,peer_output_root=payload['peer_output_root'],repo_root=repo)
            if Path(config['split']['persist_path']).resolve()!=FROZEN_SPLIT or config['split']['seed']!=42 or config['split']['allow_create'] is not False: raise RuntimeError('B_RUNTIME_SPLIT_BINDING=FAIL')
            config_path=Path(payload['config_dir'])/f'model_b_a9_seed{seed}.yaml'; save_config(config,config_path)
            previous=load(state) if state.exists() else {}; previous.update({'status':'running','current_seed':seed,'current_run_dir':None,'current_manifest_status':'not_started','current_stage':'training','epochs_completed':0,'metric_records':0,'updated_at_utc':now(),'error':None}); write(state,previous)
            log=Path(payload['logs'])/f'model_b_seed{seed}_training.log'
            env=os.environ.copy(); existing_pythonpath=env.get('PYTHONPATH',''); env['PYTHONPATH']=(f'{repo / "src"}:' f'{repo}:' f'{existing_pythonpath}')
            assert_no_competing_processes(
                f'before_seed_{seed}_launch'
            )
            with log.open('a',encoding='utf-8') as stream: code=subprocess.run([sys.executable,str(repo/'scripts/train.py'),'--config',str(config_path),'--device','cuda'],cwd=repo,stdout=stream,stderr=subprocess.STDOUT,text=True,env=env).returncode
            created=hits(root,seed,architecture)
            if code or not created: raise RuntimeError(f'training seed {seed} failed; see {log}')
            run_dir,_=created[0]; result=gate(run_dir,seed,architecture,FROZEN_SPLIT)
        previous=load(state) if state.exists() else {}; previous.update({'status':'running','current_seed':seed,'current_run_dir':str(run_dir),'completed_seeds':sorted(set(previous.get('completed_seeds',[])+[seed])),'current_manifest_status':'completed','current_stage':'completed_gate','epochs_completed':result['epochs_completed'],'metric_records':result['metric_records'],'updated_at_utc':now(),'error':None}); write(state,previous)
    final=load(state); final.update({'status':'completed','current_seed':None,'current_stage':'queue_complete','completed_at_utc':now(),'updated_at_utc':now()}); write(state,final)
if __name__=='__main__':
    try: main()
    except Exception as exc:
        state=Path(sys.argv[2]); prior=load(state) if state.exists() else {}; prior.update({'status':'failed','error':str(exc),'current_stage':'failed','updated_at_utc':now()}); write(state,prior); traceback.print_exc(); raise
'''
QUEUE_LAUNCHER.write_text(helper, encoding='utf-8')
print({'queue_state':str(QUEUE_STATE),'queue_launcher':str(QUEUE_LAUNCHER)})


## 7. Diagnóstico de recuperación — sólo lectura

No lanza, reanuda, modifica manifests ni elimina runs.

In [ ]:
def detect_live_b_processes():

    processes = subprocess.run(
        [
            'ps',
            '-eo',
            'pid=,args=',
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout

    live_processes = []

    for line in processes.splitlines():

        stripped = line.strip()

        if not stripped:
            continue

        if (
            'scripts/train.py'
            in stripped
            or
            'model_b_a9_multiseed_queue_launcher.py'
            in stripped
        ):
            live_processes.append(
                stripped
            )

    return live_processes


live = detect_live_b_processes()
state=json.loads(QUEUE_STATE.read_text()) if QUEUE_STATE.is_file() else {}
manifests=[]
for path in B_OUTPUT_ROOT.rglob('run_manifest.json'):
    try: manifests.append((path.parent,json.loads(path.read_text())))
    except Exception: raise RuntimeError(f'Manifiesto ilegible: {path}')
running=[item for item in manifests if item[1].get('status')=='running']
if live:
    RECOVERY_DIAGNOSIS,NEXT_SAFE_STEP='LIVE_PROCESSES_PRESENT','No iniciar otra cola; observar logs y estado.'
elif state.get('status')=='completed' and set(state.get('completed_seeds',[]))==set(RUN_SEEDS):
    RECOVERY_DIAGNOSIS,NEXT_SAFE_STEP='TRAINING_QUEUE_COMPLETE','No entrenar; continuar sólo con fase posterior autorizada.'
elif running:
    RECOVERY_DIAGNOSIS,NEXT_SAFE_STEP='STALE_RUNNING_RUN_AFTER_RUNTIME_LOSS','Revisar manifest/log/checkpoint; no fresh automático.'
elif any(payload.get('status') in {'failed','interrupted'} for _,payload in manifests):
    RECOVERY_DIAGNOSIS,NEXT_SAFE_STEP='INTERRUPTED_OR_FAILED_RUN','Diagnosticar el run concreto; no sobrescribir.'
else:
    RECOVERY_DIAGNOSIS,NEXT_SAFE_STEP='NO_ACTIVE_INTERRUPTED_RUN','Puede ejecutar preflight o la celda productiva tras revisión humana.'
print(json.dumps({'diagnosis':RECOVERY_DIAGNOSIS,'next_safe_step':NEXT_SAFE_STEP,'live_processes':live,'queue_state':state},indent=2,default=str))


## 8. ESTA CELDA SÍ INICIA ENTRENAMIENTO PRODUCTIVO

Sólo después de preflight y cambio deliberado de `RUN_PRODUCTIVE_QUEUE` a `True`. Ejecuta B11 → completed gate → B23 → completed gate → B37 → completed gate → B41 → completed gate → B53.

In [ ]:
if not RUN_PRODUCTIVE_QUEUE:
    print('COLA NO INICIADA: RUN_PRODUCTIVE_QUEUE=False')
else:
    if PREFLIGHT['status'] != 'PASS':

        raise RuntimeError(
            'No se inicia cola sin '
            'B_PREFLIGHT=PASS'
        )


    launch_live_processes = (
        detect_live_b_processes()
    )


    if launch_live_processes:

        print(
            'PROCESOS QUE BLOQUEAN '
            'EL LANZAMIENTO:'
        )

        for process_line in (
            launch_live_processes
        ):

            print(
                ' ',
                process_line,
            )

        raise RuntimeError(
            'B_QUEUE_LAUNCH_BLOCKED: '
            'ya existe un train.py '
            'o un queue launcher activo.'
        )


    if (
        RECOVERY_DIAGNOSIS
        != 'NO_ACTIVE_INTERRUPTED_RUN'
    ):

        raise RuntimeError(
            'B_QUEUE_LAUNCH_BLOCKED_BY_'
            'RECOVERY_STATE: '
            f'{RECOVERY_DIAGNOSIS}'
        )


    launch={'repo':str(repo),'output_root':str(B_OUTPUT_ROOT),'peer_output_root':str(A_OUTPUT_ROOT),'mutants':MUTANTS_RECORD['local_locator'],'wt':WT_RECORD['local_locator'],'architecture':ARCHITECTURE,'seeds':RUN_SEEDS,'state':str(QUEUE_STATE),'logs':str(LAUNCHER_LOG_ROOT),'config_dir':str(resolved_dir)}
    request=QUEUE_ROOT/'model_b_a9_multiseed_queue_launcher.json'; request.write_text(json.dumps(launch,indent=2),encoding='utf-8')
    launcher_log=QUEUE_ROOT/'model_b_a9_multiseed_queue.log'
    with launcher_log.open('a',encoding='utf-8') as stream: result=subprocess.run([sys.executable,str(QUEUE_LAUNCHER),str(request),str(QUEUE_STATE)],cwd=repo,stdout=stream,stderr=subprocess.STDOUT,text=True)
    if result.returncode: raise RuntimeError(f'Cola B detenida; consulte {launcher_log} y {QUEUE_STATE}')
    print('COLA B FINALIZADA. Completed gate no equivale a acceptance científica.')
